[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/en/lab2/lab2-part1.ipynb)

# Lab 1: Neural networks from scratch with NumPy

In this lab we are going to become familiar with the stochastic gradient descent algorithm with backpropagation. To do so, we will develop a classifier that uses a neural network to model the data and make predictions. This requires designing the neural network and training it so that the predictions are correct for the given dataset. We will implement everything using `numpy`.

# Prerequisites

## Install packages

For this lab we will only need `numpy` (and `pandas`, `sklearn`, and `seaborn` to load the dataset)

In [ ]:
#!pip install numpy seaborn pandas
import numpy as np
import seaborn as sns
import pandas as pd

## Creating an artificial neuron with NumPy

An artificial neuron consists of two distinct parts. First, the unit computes a sum of all its inputs (plus a *bias* component), each of them weighted by a weight. These weights (and the *bias*) are what we will modify so that the neuron produces the right output for our problem for each combination of inputs from the dataset.

Given an input vector $\mathbf{x}$ with $d$ components and a weight vector $\mathbf{w}$, this first part of the neuron will compute a single scalar output value that we will call $z$ as follows:

$$
z = \sum \limits_{i=0}^{d} \mathbf{x}_i\mathbf{w}_i + bias
$$

### Vector notation

To simplify the notation, we can represent all the weights of a neuron as a vector. By doing so, the weighted sum of the inputs becomes the dot product of the input vector $\mathbf{x}$ and the weight vector $\mathbf{w}$. Taking into account that we will assume the input vector $\mathbf{x}$ is a row vector of dimensions 1 x $d$, we will declare the weight vector $\mathbf{w}$ with the same dimensions and we can represent the dot product $\mathbf{x} \cdot \mathbf{w}$ as the matrix product $\mathbf{x}\mathbf{w}^T$. This is advantageous for processing several input vectors using the same operation.

$$
z = \sum \limits_{i=0}^{d} \mathbf{x}_i\mathbf{w}_i + bias = \mathbf{x} \cdot \mathbf{w} + bias = \mathbf{x} \mathbf{w}^T + bias
$$

Complete the following cell to compute $z$.

In [ ]:
x = np.array([1, 2, 8, -4]).reshape((1,4)) # input vector
w = np.array([0.1, -0.8, 0.3, 0.2]).reshape((1,4)) # weight vector
bias = 0.1

# TODO - complete this line using np.matmul
z = 

# Check the result
np.testing.assert_almost_equal(0.2, z, err_msg='Check your implementation')

### Activation function

After this first step, the output $z$ will be a linear combination of the inputs. If we concatenate several neurons defined this way, the result will still be a linear combination of the inputs, which is not very useful since the same result could be obtained with a single neuron. That is why we need each neuron to have a second part that introduces a non-linearity. That is what we call the *activation function*. In this example we will take the sigmoid function as the activation function, defined as:

$$sigmoid(x) = \dfrac{1}{1+e^{-x}}$$

Complete the code below to compute the sigmoid of a scalar $x$:

In [ ]:
def sigmoid(x):
    # TODO - Complete the following line
    return 

# Check the result
np.testing.assert_almost_equal(0.54983399, sigmoid(z))

In this case, we will take as input of the sigmoid function the output of the previous step, $z$. Therefore, the output $y$ of the artificial neuron will be a scalar of the following form:

$$y = sigmoid(z) = \dfrac{1}{1+e^{-(\mathbf{x}\mathbf{w}^T + bias)}}$$

You can see the general diagram of an artificial neuron in this figure:

<img src="./img/neural-model.png" alt="Diagram of an artificial neuron" width="700"/>

Complete the code of this function to perform the *forward pass* of an artificial neuron with sigmoid activation function, that is, to compute the output from the input vector, the weight vector and the bias value:

In [ ]:
def neuron_forward(x, w, bias):
    # TODO - Complete the following line
    return 

# Check
np.testing.assert_almost_equal(0.54983399, neuron_forward(x, w, bias))

## Feed-forward network

With this setup, if we used gradient descent to learn the vector $\mathbf{w}$ that makes correct predictions for a given dataset, we would be training a logistic regression model. However, the power of neural networks resides in the possibility of combining many of these units to be able to model much more complex functions, so we are going to build a network that uses several units.

The simplest way of organizing several neurons is to form a *feed-forward* network. To do this, we will first group several neurons together forming a *layer*. If the output of a neuron was a scalar $y$, the output of a layer will be a vector $\mathbf{y}$ with as many components as units the layer has. Similarly, if a neuron had a weight vector $\mathbf{w}$, a layer will have a weight matrix $\mathbf{W}$, in which each row corresponds to the weight vector of a neuron in the layer. We will also have a vector $\mathbf{b}$, where each component will be the *bias* of a neuron of the layer.

We can take advantage of NumPy's matrix and vector operations to compute the *forward pass* of a whole layer in one go. Complete the following code to obtain a function that performs the *forward pass* of a layer of artificial neurons with sigmoid activation, taking as inputs the input vector $\mathbf{x}$, a weight matrix $\mathbf{W}$ and a vector $\mathbf{b}$.

In [ ]:
# Make sure your sigmoid implementation can take a vector as input. (NumPy's np.exp is useful: it can take a scalar and return a scalar, or a vector and return a vector.)
np.testing.assert_almost_equal([0.549834, 0.98201379], sigmoid(np.array([z[0][0],4])))

def layer_forward(x, W, b):
    # TODO - Complete the code. NumPy makes this very straightforward
    return 

# Check
np.testing.assert_almost_equal(np.array([[0.549834], [0.05732418]]), layer_forward(np.vstack((x, np.array([-1, 3, -2, 2]))), w, np.array([bias, -0.1]).reshape((2,1))))

Now we can concatenate layers of neurons with ease. We are going to create a network of three layers:
 1. The layer $C_0$ has 5 units. It receives the vector $\mathbf{x}$ as input and produces the vector $\mathbf{h_0}$ as output. It has a weight matrix $\mathbf{W_0}$ and a bias vector $\mathbf{b_0}$.
 1. The layer $C_1$ has 3 units. It receives the vector $\mathbf{h_0}$ as input and produces the vector $\mathbf{h_1}$ as output. It has a weight matrix $\mathbf{W_1}$ and a bias vector $\mathbf{b_1}$.
 1. The layer $C_2$ has 1 unit. It receives the vector $\mathbf{h_1}$ as input and produces the vector $\mathbf{y}$ as output. It has a weight matrix $\mathbf{W_2}$ and a bias vector $\mathbf{b_2}$.

<img src="./img/lab1-red.png" alt="Diagram of an artificial neuron" width="700"/>

Complete the following cell to compute the output of the network.

In [ ]:
# Weight initialization
np.random.seed(1234567) # Fix the seed so random numbers match across runs and we can check results
# TODO - Complete the matrix dimensions
W0 = np.random.rand(5, x.shape[1]) - 0.5
b0 = np.random.rand(1, 5) - 0.5
W1 = np.random.rand(, ) - 0.5
b1 = np.random.rand(, ) - 0.5
W2 = np.random.rand(, ) - 0.5
b2 = np.random.rand(, ) - 0.5


# Helper that uses the network to make a prediction
def compute_prediction(x):
    # TODO - Compute each layer's output
    h0 = 
    h1 = 
    y = 
    return y

# Check
np.testing.assert_almost_equal(0.34535528, compute_prediction(x))

# Making predictions with the network on a dataset

Now that we know how to define a network, let's test what predictions it makes on a dataset. We will load the `titanic` dataset (which tries to predict the survival of Titanic passengers from their characteristics) using the `seaborn` library. With those data, we will compute the predicted output for each input and compare it with the expected label.

In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler

def load_titanic():
    # Load the Titanic dataset from seaborn
    df = sns.load_dataset('titanic')

    # 1) Select relevant variables and clean
    # Columns we will use
    cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
    df = df[cols].copy()

    # Drop rows with missing values
    df = df.dropna(subset=['age', 'embarked', 'fare'])

    # 2) Split labels and features
    y = df['survived'].to_numpy().astype(np.float32)        # labels as float
    X = df.drop(columns=['survived'])

    # 3) One-hot encoding for all categorical variables
    categorical_cols = ['pclass', 'sex', 'embarked', 'alone', 'sibsp', 'parch']
    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)  # drop_first=True avoids multicollinearity

    # 4) Numeric variables
    numeric_cols = ['age', 'fare']
    X_numeric = X_encoded[numeric_cols + [c for c in X_encoded.columns if c not in numeric_cols]]
    scaler = StandardScaler()
    X_numeric[numeric_cols] = scaler.fit_transform(X_numeric[numeric_cols])

    # 5) Convert to numpy arrays
    X_np = X_numeric.to_numpy().astype(np.float32)
    y_np = y.reshape(-1, 1).astype(np.float32)  # reshape to (n_samples, 1)

    return X_np, y_np

In [ ]:
X, y = load_titanic()

# TODO - Adapt the input layer (C0) parameter shapes to the dataset vectors, which have 24 components
W0 = np.random.rand(, ) - 0.5
b0 = np.random.rand(, )


# Helper that, given input vectors, computes the network accuracy
def compute_accuracy(examples_x, examples_y):
    num_hits = 0
    num_samples = 0
    for x, label in zip(examples_x, examples_y):
        y_pred = compute_prediction(x)

        # Update the element and hit counters
        num_samples += 1
        if ((y_pred > 0.5) and (label==1)) or ((y_pred <= 0.5) and (label==0)):
            num_hits += 1

    return num_hits / float(num_samples)
        
print('Accuracy is', compute_accuracy(X[:100], y[:100]))

Predictably, the network does not work well because the weight matrices contain random vectors. For the predictions to improve, we have to fit our model to the dataset, that is, find values for the parameters ($\mathbf{W_0}$, $\mathbf{b_0}$, $\mathbf{W_1}$, $\mathbf{b_1}$, $\mathbf{W_2}$, $\mathbf{b_2}$) that yield good predictions.

# Training the network

## The cost function

To train the network we will use an optimization process. First, we must define which function we want to optimize. We need a function that, given a combination of network parameters, returns a high value when the predictions with those parameters are bad and a low value when they are good. That is what we call the **cost function** ($J$). We will define a **loss function** ($\mathcal{L}$) that receives as input a prediction and a real label and tells us how wrong the prediction is. In this case we will use the binary cross entropy, described as:

$$ \mathcal{L}(y_{pred},y_{label}) = - y_{label} \log(y_{pred}) - (1-y_{label})  \log(1-y_{pred}) $$

The cost function ($J$) will be the average of the loss function over the $m$ examples of the training set:
$$ J(\mathbf{W},\mathbf{b}) = \frac{1}{m} \sum_{i=1}^m \mathcal{L}(y_{pred}^{(i)}, y_{label}^{(i)})$$

If we minimize $J$, our predictions will be better. To minimize $J$ we will use **gradient descent**: we will take successive steps in which we compute the gradient of $J$ with respect to the different parameters ($\mathbf{W},\mathbf{b}$) and update the parameters in the direction of the gradient, hoping that the next step obtains a smaller value of $J$. We will repeat this process for a fixed number of steps.

Therefore, the algorithm we must apply is the following:
 1. Compute the loss of the predictions with the current values of $\mathbf{W}$ and $\mathbf{b}$
 1. Compute the gradient with respect to $\mathbf{W}$ and $\mathbf{b}$.
 1. Update $\mathbf{W}$ and $\mathbf{b}$ in the direction of their respective gradients.

## Gradients

The second step forces us to be able to compute the gradient of $J(\mathbf{W},\mathbf{b})$ with respect to each of the parameters, that is, the partial derivative of $J(\mathbf{W},\mathbf{b})$ with respect to each parameter. To do this, we will propagate the gradient backwards, computing at each step the gradient at the previous node from the subsequent ones. Let's compute, for example, the gradient of $J(\mathbf{W},\mathbf{b})$ with respect to $z_2$:

$$ \frac{\partial J(\mathbf{W},\mathbf{b})}{\partial z_2} = \frac{\partial (\frac{1}{m} \sum_{i=1}^m \mathcal{L}(y_{pred}^{(i)}, y_{label}^{(i)}))}{\partial z_2} = \frac{1}{m} \sum_{i=1}^m \frac{\partial \mathcal{L}(y_{pred}^{(i)}, y_{label}^{(i)})}{\partial z_2} $$

Applying the chain rule ([see the full development](./lab2-gradientes.pdf)) we obtain the following formulas for the gradients:
$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial b_2} = \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial z_2}$$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial W_2} = \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial z_2} h_1^T $$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial h_1} = \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial z_2} W_2^T $$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial z_1} = \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial h_1} h_1 (1 - h_1) = dLdz1$$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial b_1} = \sum_{i=1}^{5} dJdz1_i $$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial W_1} = diag(dLdz1) h_0^T $$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial h_0} = diag(dLdz1) W_1 $$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial z0} = \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial h_0} h_0 (1 - h_0) = dLdz0$$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial b_0} = \sum_{i=1}^{5} dLdz0_i $$

$$ \frac{\partial \mathcal{L}(y_{pred}, y_{label})}{\partial W_0} = diag(dLdz0) x^T $$

Knowing these formulas we can adapt our function so that, in addition to computing the prediction, it returns the gradients of the loss function with respect to the variables involved.

In [ ]:
def compute_and_propagate(x, y_label):
    ''' Returns the predicted output y, the loss value, and a dictionary of gradients
    '''
    x = np.expand_dims(x, axis=0)
    
    # Forward pass
    # TODO - Fill in the following lines
    z0 = 
    h0 = 
    z1 = 
    h1 = 
    z2 = 
    y = 
    
    #Backpropagation
    dLdz2 = y - y_label
    dLdb2 = dLdz2
    dLdW2 = dLdz2 * h1
    dLdh1 = np.matmul(dLdz2.T, W2)
    dLdz1 = dLdh1 * h1 * (1 - h1)
    dLdb1 = dLdz1
    dLdW1 = np.matmul(dLdz1.T, h0)
    dLdh0 = np.matmul(dLdz1, W1)
    dLdz0 = dLdh0 * h0 * (1 - h0)
    dLdb0 = dLdz0
    dLdW0 = np.matmul(dLdz0.T, x)
    
    # Gradients must have the same shape as the variables they refer to
    assert(dLdz2.shape==z2.shape)
    assert(dLdb2.shape==b2.shape)
    assert(dLdW2.shape==W2.shape)
    assert(dLdh1.shape==h1.shape)
    assert(dLdz1.shape==z1.shape)
    assert(dLdb1.shape==b1.shape)
    assert(dLdW1.shape==W1.shape)
    assert(dLdh0.shape==h0.shape)
    assert(dLdz0.shape==z0.shape)
    assert(dLdb0.shape==b0.shape)
    assert(dLdW0.shape==W0.shape)
    
    # Build the gradient dictionary
    gradients = {}
    gradients["b2"] = dLdb2
    gradients["b1"] = dLdb1
    gradients["b0"] = dLdb0
    gradients["W2"] = dLdW2
    gradients["W1"] = dLdW1
    gradients["W0"] = dLdW0
    
    # Loss
    # TODO - Fill in the following line
    loss = 
    return y, loss.flatten(), gradients

# verificación
prediction, loss, gradients = compute_and_propagate(X[0], y[0])
np.testing.assert_almost_equal(0.3463054, prediction[0])
np.testing.assert_almost_equal(0.4251149366689026, loss[0])

## Learning loop

Once we are able to compute the gradients with respect to the variables for each example of the training set, we can already implement the training loop.

Complete this function so that it performs `num_steps` of training in which the following is done:
 1. Compute predictions, losses, and gradients for each element of the training set
 1. Compute the average value of each gradient
 1. Use those averages to update the variables

We will also keep track of the accuracy and the value of the cost function at each step.

In [ ]:
def train(examples, labels, num_steps, learning_rate = 0.01):
    step = 0
    while step < num_steps:
        num_hits = 0
        num_samples = 0
        total_loss = 0
        for x, label in zip(examples, labels):
            # TODO - Complete the following line
            y_pred, loss, gradients = 

            # Update element and hit counters
            num_samples += 1
            if ((y_pred > 0.5) and (label==1)) or ((y_pred <= 0.5) and (label==0)):
                num_hits += 1
                
            # Update the total-loss accumulator
            total_loss += loss
            
            # Update variables in the direction of their gradient
            global W2, W1, W0, b2, b1, b0
            # TODO - Complete the following lines
            W2 = W2 - learning_rate * gradients['W2']
            W1 = 
            W0 = 
            b2 = 
            b1 = 
            b0 = 
            
        accuracy = num_hits / float(num_samples)
        loss = total_loss / float(num_samples)
        print(f'Epoch: {step}/{num_steps}: Loss: {loss} Accuracy: {accuracy}')
        step += 1

To test our training algorithm, let's take only the first 100 elements of the dataset and train the model to fit those data.

In [ ]:
batch_size = 100
train(X[:batch_size], y[:batch_size], 5000)

Implementing the training loop is a complicated task because each iteration requires the *forward propagation* and the *backpropagation* which, in turn, involves keeping track of intermediate results to compute each of the gradients with which the parameters are updated.

In addition to the complexity of organizing the code that carries out that algorithm, the following factors must be taken into account:
 - The code should allow networks with any number of layers which, in turn, should contain an arbitrary number of units. It would even be ideal if it allowed architectures different from the *feed forward* network.
 - Memory usage must be efficient, not storing more results than needed for each computation.
 - Likewise, it should avoid performing unnecessary operations and parallelize those that can be.
 - The code should be extensible to allow other cost functions.

That is why specialized frameworks such as TensorFlow are commonly used to work with neural networks. This type of software allows us to define models and train them efficiently without having to worry about computing derivatives or the implementation details.